<div style="background:linear-gradient(135deg,#0b2545 0%,#13315c 55%,#1f7a8c 100%);color:#ffffff;border-radius:18px;padding:44px 40px;margin:0 0 18px 0;">
<div style="font-size:13px;letter-spacing:2.2px;text-transform:uppercase;opacity:.82;">RSNA Knee Abnormality Detection &middot; solution analysis</div>
<div style="font-size:34px;line-height:1.15;font-weight:800;margin:12px 0 14px 0;">🦵  RSNA Knee 0.957 Explained: Why the Open Ensemble Works and Where It Stops</div>
<div style="font-size:17px;line-height:1.55;opacity:.93;max-width:820px;">An analysis of the leading open solutions: the data, where the labels come from, what the score measures, how each model reads a study, how their opinions are combined, what each change to the base recipe adds, and the submitted notebook that scores 0.957 on the public leaderboard.</div>
</div>

<div style="border:1px solid rgba(31,122,140,.45);background:rgba(31,122,140,.07);border-radius:16px;padding:6px 28px 10px 28px;margin:14px 0;">

<div style="font-size:12px;letter-spacing:1.8px;text-transform:uppercase;opacity:.7;margin-top:14px;">About this article</div>

**The question.** The best public notebooks of this challenge all score between 0.956 and 0.957, and 940 teams print exactly 0.956. They are built from the same few published models. How do these solutions work, why does averaging their opinions help, how far can it go, and what can a leaderboard actually tell us about a change of a few ten-thousandths?

**What the article contains**

* A handbook of 56 pages, readable in this notebook: the data, the labels read out of radiology reports, the score and its noise, the published models, the blend, the comparison of the recipes, and what limits them.
* The math behind the score and behind averaging, with code that draws the figures.
* The submitted notebook, unchanged, in the last part. It is the notebook version that printed 0.957.

**Where this version stands.** On the public leaderboard of 15 October 2026 (6,035 teams) the base recipe copied unchanged prints 0.956, and this notebook prints 0.957, place 206.

</div>

## 📖 The handbook

The handbook is an analysis, not a diary. Every number is marked as *computed* (from the competition files or by running the notebooks), *reported* (published by an author) or *estimate* (from a formula or simulation under stated assumptions). It contains no scans, because the competition data may not be republished; the figures are aggregate statistics, original schematics, and calculations.

📦 Dataset with the handbook PDF and the reader: [kaggle.com/datasets/leoprovorov/rsna-knee-ensemble-analytical-handbook](https://www.kaggle.com/datasets/leoprovorov/rsna-knee-ensemble-analytical-handbook)

👇 **Read it here.** Scroll inside the frame. The menu on the left jumps between chapters, **A− / A+** change the text size, **◐** switches light, dark and sepia themes, and a click on a figure enlarges it.

In [ ]:
# The handbook reader is the file handbook_reader.html of the attached dataset.
import glob
import html
import os
from IPython.display import HTML, display

candidates = [os.environ.get("HANDBOOK_READER", "")] + sorted(glob.glob("/kaggle/input/**/handbook_reader.html", recursive=True))
path = next((p for p in candidates if p and os.path.exists(p)), None)
if path:
    reader = open(path, encoding="utf-8").read()
    display(HTML(
        '<iframe srcdoc="' + html.escape(reader, quote=True) + '" width="100%" height="820" '
        'style="border:1px solid rgba(128,128,128,.35);border-radius:12px;background:#fff"></iframe>'))
else:
    display(HTML("<p>The reader is part of the handbook dataset. Attach it with <b>Add Input</b>, then run this cell again.</p>"))

### Contents of the handbook

**Introduction.** What the challenge asks, what the book contains, how numbers are marked.

**Part I &middot; The task**

| Chapter | What it covers | Page |
|---|---|---|
| 1. The examinations and the twelve findings | A study, its series and planes, what the twelve findings mean, scanners and file formats, how a submission is scored. | 6 |
| 2. Where the training labels come from | Reports turned into soft labels, how close they are to expert labels (AUC about 0.89), why a model can beat its labels. | 12 |
| 3. The score: ROC AUC and how noisy it is | AUC from a six-study example to a simulation; noise of the score; what a leaderboard can resolve. | 19 |

**Part II &middot; The models**

| Chapter | What it covers | Page |
|---|---|---|
| 4. The published models and how each one reads a study | Raptor (CoAtNet), R7 and S6 (ConvNeXt), the 2.5D reader, the anatomical mirror. | 26 |
| 5. Combining the models | Rank averaging, the blend, the ceiling of a family of models, a check on the 58 expert-labelled studies. | 31 |

**Part III &middot; The recipes compared**

| Chapter | What it covers | Page |
|---|---|---|
| 6. The leading recipes and what each change adds | The base recipe, the changes, the earlier generation, running time, measured scores. | 39 |
| 7. Where the recipes stand on the leaderboard | The table, the spike at 0.956, the place inside a block of equal scores. | 43 |
| 8. How stable is the ranking on the hidden half | A shake-up simulation; the value of two final submissions. | 46 |
| 9. What limits the score and what could raise it | Other models, other information, better labels, the cost of training. | 49 |

**Appendices.** Sources and credits, where the numbers come from, glossary.

## 🧮 Two pieces of math behind everything else

### 1 &middot; What the score measures, and how much it moves by chance

For one finding, let $S^+$ be the score a model gives to a random study that has the finding and $S^-$ the score for a random study without it. The ROC AUC is

$$ \mathrm{AUC}=P(S^+>S^-)+\tfrac12 P(S^+=S^-), \qquad \text{Score}=\frac1{12}\sum_{k=1}^{12}\mathrm{AUC}_k . $$

Only the *order* of the studies counts, so any strictly increasing change of the scores leaves the AUC unchanged. That is why models are combined by averaging **ranks**. The AUC measured on $n_+$ positive and $n_-$ negative studies has the standard error of Hanley and McNeil (1982), with $A$ the true AUC, $Q_1=\frac{A}{2-A}$ and $Q_2=\frac{2A^2}{1+A}$:

$$ \mathrm{SE}(A)=\sqrt{\frac{A(1-A)+(n_+-1)(Q_1-A^2)+(n_--1)(Q_2-A^2)}{n_+n_-}} . $$

The next cell computes it for a public half of about 640 studies, and simulates the 12-finding score.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, rankdata
import warnings
warnings.filterwarnings("ignore")

BLUE, ORANGE, GREEN, VERM = "#0072B2", "#E69F00", "#009E73", "#D55E00"   # colour-blind-safe palette

def hanley_se(A, n1, n0):
    q1, q2 = A / (2 - A), 2 * A**2 / (1 + A)
    return np.sqrt((A*(1-A) + (n1-1)*(q1-A**2) + (n0-1)*(q2-A**2)) / (n1*n0))

# finding rates of the report-derived labels (reader 1 of the handbook, chapter 2)
PREV = np.array([0.208, 0.153, 0.404, 0.157, 0.372, 0.273, 0.457, 0.598, 0.127, 0.247, 0.175, 0.072])
rng = np.random.default_rng(2026)

def auc_fast(score, y):
    r = rankdata(score); n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def macro_sd(N, auc_true=0.957, trials=300):
    d = np.sqrt(2) * norm.ppf(auc_true)
    out = []
    for _ in range(trials):
        a = []
        for p in PREV:
            y = (rng.random(N) < p).astype(int)
            a.append(auc_fast(rng.normal(0, 1, N) + d * y, y))
        out.append(np.mean(a))
    return np.std(out)

Ns = [320, 640, 1284, 2568]
sds = [macro_sd(N) for N in Ns]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
k = np.arange(5, 400)
for A, ls, c in ((0.90, ":", BLUE), (0.95, "-", ORANGE), (0.97, "--", GREEN)):
    ax[0].plot(k, [hanley_se(A, i, 640 - i) for i in k], ls=ls, lw=2.4, color=c, label=f"AUC = {A}")
ax[0].set_xlabel("positive studies among 640"); ax[0].set_ylabel("standard error of one finding's AUC"); ax[0].legend(); ax[0].set_title("One finding, 640 studies")
bars = ax[1].bar([str(n) for n in Ns], sds, color=[BLUE, ORANGE, GREEN, VERM], edgecolor="black")
for b, h in zip(bars, ["...", "///", "\\\\", "xxx"]):
    b.set_hatch(h)
for i, s in enumerate(sds):
    ax[1].text(i, s + 0.00008, f"{s:.4f}", ha="center")
ax[1].set_xlabel("studies scored"); ax[1].set_ylabel("sd of the 12-finding score"); ax[1].set_title("Macro score of 12 findings (true 0.957 each)")
plt.tight_layout(); plt.show()
print({N: round(s, 4) for N, s in zip(Ns, sds)})

**How to read the figure.** *Left:* the horizontal axis is the number of studies with the finding among 640; the vertical axis is the standard error of the AUC from the formula. Rare findings (few positives) are measured much less precisely than common ones; the three curves are for true AUC 0.90 (dotted), 0.95 (solid) and 0.97 (dashed). *Right:* the standard deviation of the 12-finding score over simulated test sets of different sizes (bars carry different hatching in addition to colour). For about 640 studies it is about 0.003, which is larger than the whole spread of the top 400 teams (0.0022). So a leaderboard cannot confirm a gain of a few ten-thousandths.

### 2 &middot; Why averaging helps, and why it stops helping

Take one finding. Let every model's score be the true signal plus noise, $s_m=d\,y+\varepsilon_m$, with noise variance 1 and correlation $\rho$ between the noises of two models. One model has $\mathrm{AUC}_1=\Phi(d/\sqrt2)$. The average of $n$ models has noise variance $\frac{1+(n-1)\rho}{n}$, so

$$ \mathrm{AUC}_n=\Phi\!\left(\frac{d}{\sqrt2\,\sqrt{\frac{1+(n-1)\rho}{n}}}\right),\qquad \mathrm{AUC}_\infty=\Phi\!\left(\frac{d}{\sqrt{2\rho}}\right). $$

The part $\rho$ of the noise is shared by all models and no number of models removes it. Solving for $\rho$ from published scores (single models about 0.950, a blend of two at 0.956, a blend of four at 0.957) gives the shared-error level of the open models and the ceiling of the family.

In [ ]:
def auc_n(n, rho, auc1=0.95):
    d = np.sqrt(2) * norm.ppf(auc1)
    return norm.cdf(d / (np.sqrt(2) * np.sqrt((1 + (n - 1) * rho) / n)))

def implied_rho(n, auc1, aucn):
    d = np.sqrt(2) * norm.ppf(auc1)
    k = (d / (np.sqrt(2) * norm.ppf(aucn))) ** 2          # noise variance of the average
    return (k * n - 1) / (n - 1)

r2, r4 = implied_rho(2, 0.950, 0.956), implied_rho(4, 0.950, 0.957)
print(f"implied error correlation: two models 0.950 -> 0.956: rho = {r2:.2f};  four models 0.950 -> 0.957: rho = {r4:.2f}")
for rho in (r2, r4):
    print(f"rho = {rho:.2f}: ceiling for many models = {norm.cdf(np.sqrt(2) * norm.ppf(0.95) / np.sqrt(2 * rho)):.4f}")

ns = np.arange(1, 13)
fig, ax = plt.subplots(figsize=(7.6, 4.3))
for rho, ls, c, mk in ((0.7, "-", BLUE, "o"), (0.8, "--", ORANGE, "s"), (0.86, "-.", GREEN, "^"), (0.92, ":", VERM, "D")):
    ax.plot(ns, [auc_n(n, rho) for n in ns], ls=ls, color=c, marker=mk, lw=2.3, label=f"shared-error correlation rho = {rho}")
ax.set_xlabel("number of equally good models averaged (each 0.950 alone)"); ax.set_ylabel("AUC of the average")
ax.set_title("More models help less and less"); ax.legend(); plt.tight_layout(); plt.show()

**How to read the figure.** The horizontal axis is the number of equally good models in the average and the vertical axis the AUC of the average. Each curve is one value of the shared-error correlation $\rho$ (solid 0.7, dashed 0.8, dash-dotted 0.86, dotted 0.92; the markers differ too). The higher $\rho$ is, the lower the curve and the earlier it flattens. The published blends sit at $\rho\approx0.86$ to $0.89$, where eight models would reach about 0.960. These are estimates: they treat all models as equally good with equally correlated errors.

### 3 &middot; Key results of the handbook

Four figures from the handbook, with the numbers behind them.

**Report labels are an imperfect teacher.** Labels read out of the radiology reports by a language model reproduce the 58 expert labels with a macro AUC of 0.89 (95% interval 0.86 to 0.92), yet the models trained on them score about 0.957 on the hidden test. Noise in the labels that does not depend on the image does not change the best possible ranking (chapter 2).

In [ ]:
import glob, os
from IPython.display import Image, display

def show(name, width=980):
    cands = sorted(glob.glob("/kaggle/input/datasets/leoprovorov/rsna-knee-ensemble-analytical-handbook/figures/" + name, recursive=True))
    if cands:
        display(Image(filename=cands[0], width=width))
    else:
        print("figure", name, "is part of the handbook dataset: attach it with Add Input")

show("fig_2_3_labels_vs_expert.png")

**How to read the figure.** Each pair of bars is one finding. The vertical axis is the AUC of the report-derived label against the expert label on the 58 expert-labelled studies (Reader 1, solid blue, 58 studies; Reader 2, orange with diagonal hatching, 57 studies). The dashed red line marks 0.957, the score of the best public ensembles on the hidden test set.

**Five models, one family of labels.** The published models differ in network, in how they choose the slices and in extra data, but all learned from report-derived labels (chapter 4).

In [ ]:
show("fig_4_3_members.png")

**How to read the figure.** Each row is one model: the network, what it looks at, how many trained models stand behind it, the extra training signal, and the public score reported by its author.

**What the models do on the 58 expert-labelled studies.** The published models were run on the 58 studies and their blends computed with the same weights as in the submitted notebook. R7 and S6 were trained without these studies; for the Raptor models the author reports that the 58 were used to select them, so their numbers are optimistic. Left: macro AUC with a 95% bootstrap interval. Right: rank correlation between the models.

In [ ]:
show("fig_5_2_gold58.png")

**How to read the figure.** *Left:* each row is a model (blue circles) or a blend (red diamonds); the dot is the macro AUC on the 58 studies and the bar is the 95% bootstrap interval. *Right:* the average rank correlation of the models over the twelve findings; darker is more similar. With 58 studies the intervals are wide (about ±0.03), which is why the leaderboard and not these 58 studies is used for the final comparison.

In [ ]:
show("fig_7_1_leaderboard.png")

**How to read the figure.** *Left:* teams per printed score on a logarithmic axis; blue below 0.950, orange 0.950 to 0.956, red hatched from 0.957. *Right:* teams at or above each printed score; counts are printed for 0.956 to 0.967.

# RSNA Knee v2: rank blend of Raptor-384, Raptor-224 (plain + mirror), R7 ConvNeXt and S6 ConvNeXt (all data)

Private working notebook. Every model is a public checkpoint and every stage is a public script run as published, with only the glue changed:

- Stage 1 - nartaa's Raptor CoAtNet-384 (0.949, plain + anatomical-mirror view), unchanged.
- Stage 1b - nartaa's Raptor CoAtNet-224 crop (0.945) with the anatomical-mirror view added (the `[s9]` change from varshithakolupuri's notebook).
- Stage 2 - heliosli's compact ConvNeXt-small R7 (4 folds), unchanged.
- Stage 2b - nartaa's S6 ConvNeXt-small trained on all data (0.950 alone), nartaa's published script with the Raptor-384 recomputation replaced by the stage-1 output; optional: if it fails or time is short the blend uses the other members.
- Stage 3 - per-finding percentile-rank blend with weights fixed before any run: Raptor half = 0.65 x rank(384) + 0.35 x rank(224); compact half = 0.60 x rank(R7) + 0.40 x rank(S6); final = 0.5 / 0.5. Nothing is fitted on labels.

## Credits and required acknowledgement

Built on dreaddevelopment's [Raptor CoAtNet/MIL inference recipe and public baseline weights](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense). These are our retrained checkpoints, rather than those baseline weights. Weak-label sources include [stevenleehans](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels), [pilkwang](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels), and [riadmohamed42's JEV labels and verification](https://www.kaggle.com/code/riadmohamed42/jev-knee-labels-verification), combined with our report-derived labels. Source reports and study-level label tables are not included here. The CoAtNet backbone comes from timm. Competition data is provided under the [RSNA Knee competition rules](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/rules).

Data and/or research tools used in the preparation of this manuscript were obtained and analyzed from the controlled access datasets distributed from the Osteoarthritis Initiative (OAI), a data repository housed within the NIMH Data Archive (NDA). OAI is a collaborative informatics system created by the National Institute of Mental Health and the National Institute of Arthritis, Musculoskeletal and Skin Diseases (NIAMS) to provide a worldwide resource to quicken the pace of biomarker identification, scientific investigation and OA drug development. Dataset identifier(s): 10.15154/0hcg-f676.

[Shared NDA Study](https://nda.nih.gov/study.html?id=3407) · [DOI: 10.15154/0hcg-f676](https://doi.org/10.15154/0hcg-f676). Official acknowledgement source: [NDA manuscript preparation](https://nda.nih.gov/nda/manuscript-preparation).


The 224-pixel stage and its share of the Raptor half follow [varshithakolupuri's notebook](https://www.kaggle.com/code/varshithakolupuri/rsna-knee-one-unused-checkpoint-lifts-the-0-956) (35 % of the Raptor slot); both checkpoints and both inference scripts are nartaa's public work.

## 🤝 Let&rsquo;s think about this together

**Ideas are welcome.** Read it, pick it apart, tell me where it is wrong.

### Questions I would love your help with

1. **Is the ceiling of about 0.96 real?** It is derived from two blends and an equal-correlation model. Has anyone measured the errors of the models on a larger set of expert-labelled studies?
2. **Better readers of the reports.** The report labels agree with experts at an AUC of about 0.89. Which prompt, model or combination of readings gets closer on the 58 expert studies, and does it carry over to the models?
3. **Why are some findings hard?** Synovitis, contusion and lateral osteoarthritis agree least with the experts. Is the cause the reports or the images?
4. **Does the mirror pass help the compact models?** Is a gain of a few ten-thousandths visible to anyone on a larger set?

### Credits

* **dreaddevelopment** (Raptor recipe and soft labels), **nartaa** (Raptor 384 and 224 checkpoints, S6 weights, label tables), **heliosli** (R7, the code and anatomy network, the base recipe), **varshithakolupuri** (224 checkpoint and mirror view in the base recipe), **goodpjw2008** (2.5D reader), **bliverrigpdoge**, **pilkwang**, **stevenleehans**, **riadmohamed42**, **mattiaangeli** and **prvsiyan**. Details and links are in appendix A of the handbook.
* The Osteoarthritis Initiative acknowledgement required by two of the models is in appendix A of the handbook.

Third-party code and model weights remain subject to their respective licenses.